# Notebook 01 — Bronze · Ingestão

**Objetivo:** Ler os arquivos CSV brutos do Volume e salvar como tabelas Delta sem nenhuma transformação.  
**Fonte:** `/Volumes/workspace/conciliacao/raw_data/`  
**Destino:** Schema `bronze`  
**Autor:** Davi Alves

In [0]:
# Cria o schema bronze se ainda não existir
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
print("Schema bronze criado com sucesso")

Schema bronze criado com sucesso


In [0]:
from pyspark.sql.functions import current_timestamp

# Caminhos dos arquivos no Volume
path_transacoes   = "/Volumes/workspace/conciliacao/raw_data/raw_transacoes.csv"
path_contas       = "/Volumes/workspace/conciliacao/raw_data/raw_contas.csv"
path_fornecedores = "/Volumes/workspace/conciliacao/raw_data/raw_fornecedores.csv"

# Lê os CSVs como string pura — sem inferir tipos, preserva o dado bruto
df_transacoes   = spark.read.option("header", True).option("inferSchema", False).csv(path_transacoes)
df_contas       = spark.read.option("header", True).option("inferSchema", False).csv(path_contas)
df_fornecedores = spark.read.option("header", True).option("inferSchema", False).csv(path_fornecedores)

# Adiciona coluna de controle com timestamp da ingestão
df_transacoes   = df_transacoes.withColumn("dt_carga", current_timestamp())
df_contas       = df_contas.withColumn("dt_carga", current_timestamp())
df_fornecedores = df_fornecedores.withColumn("dt_carga", current_timestamp())

# Salva como tabelas Delta no schema bronze
df_transacoes.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.transacoes")
df_contas.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.contas")
df_fornecedores.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.fornecedores")

print("Bronze carregado com sucesso")
print(f"transacoes:   {df_transacoes.count()} linhas")
print(f"contas:       {df_contas.count()} linhas")
print(f"fornecedores: {df_fornecedores.count()} linhas")

Bronze carregado com sucesso
transacoes:   3030 linhas
contas:       8 linhas
fornecedores: 10 linhas


## Relatório de carga — Bronze

| Tabela | Linhas | Colunas | Status |
|---|---|---|---|
| bronze.transacoes | 3.030 | 13 (12 originais + dt_carga) | ✅ Carregado |
| bronze.contas | 8 | 6 (5 originais + dt_carga) | ✅ Carregado |
| bronze.fornecedores | 10 | 5 (4 originais + dt_carga) | ✅ Carregado |

**Observação:** Dados preservados em formato bruto (todas as colunas como string).  
Nenhuma transformação aplicada. Tipagem e limpeza serão realizadas no Notebook 02 — Silver.